# 🎬 AnimeStudio AI — Kaggle GPU Worker

Free Kaggle GPU as a render worker for your AnimeStudio AI studio.

## Setup
1. New Kaggle notebook → **Settings → Accelerator → GPU** (P100 or T4×2).
2. Turn **Internet ON** (right-hand panel) — needed only for Mode B (server).

## Mode A — Standalone (works TODAY, default)
1. Edit the **Config** cell: paste your scene `PROMPT`, pick `STYLE` / `TYPE`.
2. **Run all**, keep the tab open.
3. The MP4 is saved to `/kaggle/working/output.mp4` → download it from the
   notebook's output file panel → import into the studio's **Library** page.

## Mode B — Server queue (later, after the Vercel backend exists)
Add secrets `ANIMESTUDIO_URL` + `ANIMESTUDIO_WORKER_TOKEN` (**Add-ons →
Secrets**, attach to this notebook), set `MODE = "server"` in the Config cell,
and run the **Server loop** cell — it polls `/api/jobs/next`
(see `docs/api-spec.md`).

In [ ]:
# Install dependencies (runs once per session)
!pip install -q diffusers transformers accelerate safetensors imageio imageio-ffmpeg requests Pillow
print("deps installed")

In [ ]:
# ================= CONFIG — EDIT ME =================
MODE = "standalone"   # "standalone" (today) or "server" (after Vercel backend)

# --- Standalone mode: paste your scene here ---
PROMPT = "Slow aerial drift over a smouldering crater at night, embers rising"
STYLE = "cinematic"      # "cinematic" or "anime"
TYPE = "text-to-video"   # "text-to-video" or "image-to-video"
# For image-to-video, upload the start frame via the input panel (see next cell).

# --- Server mode secrets (only when MODE = "server") ---
# Add-ons → Secrets: ANIMESTUDIO_URL, ANIMESTUDIO_WORKER_TOKEN (attached)
WORKER_NAME = "kaggle"
POLL_INTERVAL = 60
# ========================================================
print("config ok ✅  mode =", MODE)

In [ ]:
import torch
from diffusers import LTXPipeline, LTXImageToVideoPipeline

MODEL_ID = "Lightricks/LTX-Video"  # 2B params — fits a free T4 with CPU offload
DTYPE = torch.bfloat16

print("Loading text-to-video pipeline …")
t2v = LTXPipeline.from_pretrained(MODEL_ID, torch_dtype=DTYPE)
t2v.enable_model_cpu_offload()

print("Loading image-to-video pipeline …")
i2v = LTXImageToVideoPipeline.from_pretrained(MODEL_ID, torch_dtype=DTYPE)
i2v.enable_model_cpu_offload()

print("Model ready ✅")

In [ ]:
import io
import requests
from PIL import Image
from diffusers.utils import export_to_video


def style_suffix(style: str) -> str:
    return ("japanese anime style, vibrant cel shading, clean line art"
            if style == "anime"
            else "photorealistic cinematic film still, natural lighting")


def render(prompt: str, style: str = "cinematic", type: str = "text-to-video",
           start_image=None, out_path: str = "/tmp/output.mp4") -> str:
    """Render one clip and save it to out_path. Returns out_path."""
    full_prompt = prompt + ", " + style_suffix(style)
    print("Prompt:", full_prompt[:200], "…")
    if type == "image-to-video" and start_image is not None:
        img = Image.open(start_image).convert("RGB").resize((768, 512))
        frames = i2v(prompt=full_prompt, image=img,
                     num_frames=121, num_inference_steps=30,
                     guidance_scale=3.0).frames[0]
    else:
        frames = t2v(prompt=full_prompt, height=512, width=768,
                     num_frames=121, num_inference_steps=30,
                     guidance_scale=3.0).frames[0]
    export_to_video(frames, out_path, fps=24)
    print("saved →", out_path)
    return out_path


print("render() ready ✅")

In [ ]:
assert MODE == "standalone", "Set MODE = 'standalone' in the Config cell for this cell."

start_img = None
if TYPE == "image-to-video":
    import os as _os
    cands = [f"/kaggle/input/{d}/{f}" for d in _os.listdir("/kaggle/input")
             for f in _os.listdir(f"/kaggle/input/{d}")]
    assert cands, ("Upload the start frame first: notebook input panel → "
                   "'Upload' a dataset/image, then re-run.")
    print("using start frame:", cands[0])
    start_img = cands[0]

out = render(PROMPT, STYLE, TYPE, start_image=start_img,
             out_path="/kaggle/working/output.mp4")
print("⬇ Download /kaggle/working/output.mp4 from the output file panel,")
print("   then import it into the studio's Library page ✅")

## Mode B — Server queue loop (future Vercel backend)

Only run this cell when `MODE = "server"`, the secrets are attached, and the
backend from `docs/api-spec.md` is deployed.

In [ ]:
# Server-mode credentials from Kaggle Secrets (only used when MODE = "server")
if MODE == "server":
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    WEBSITE_URL = _s.get_secret("ANIMESTUDIO_URL")
    WORKER_TOKEN = _s.get_secret("ANIMESTUDIO_WORKER_TOKEN")
    assert WEBSITE_URL and WORKER_TOKEN, "Attach the ANIMESTUDIO_URL and ANIMESTUDIO_WORKER_TOKEN secrets!"
    print("server credentials ok ✅")
else:
    WEBSITE_URL, WORKER_TOKEN = "", ""
    print("standalone mode — skipping server credentials")

In [ ]:
import time
import requests

BASE = WEBSITE_URL.rstrip("/")
HEADERS = {"Authorization": f"Bearer {WORKER_TOKEN}"}


def poll_next():
    r = requests.get(f"{BASE}/api/jobs/next", params={"worker": WORKER_NAME},
                     headers=HEADERS, timeout=30)
    if r.status_code == 204:
        return None
    r.raise_for_status()
    return r.json()["job"]


def report_complete(job_id: str, path: str):
    with open(path, "rb") as f:
        files = {"video": ("output.mp4", f, "video/mp4")}
        r = requests.post(f"{BASE}/api/jobs/{job_id}/complete",
                          headers=HEADERS, files=files, timeout=600)
    r.raise_for_status()


def report_fail(job_id: str, err: Exception):
    try:
        requests.post(f"{BASE}/api/jobs/{job_id}/fail",
                      headers={**HEADERS, "Content-Type": "application/json"},
                      json={"error": str(err)[:500]}, timeout=30)
    except Exception as e2:
        print("Could not report failure:", e2)


print("server helpers ready ✅")

In [ ]:
assert MODE == "server", "Set MODE = 'server' in the Config cell to run the queue loop."
assert WEBSITE_URL and "your-app" not in WEBSITE_URL, "Paste your real website URL in the Config cell."
assert WORKER_TOKEN, "Paste your WORKER_TOKEN in the Config cell."

print(f"Worker '{WORKER_NAME}' polling {WEBSITE_URL} every {POLL_INTERVAL}s …")
while True:
    try:
        job = poll_next()
    except Exception as e:
        print("poll error:", e)
        time.sleep(POLL_INTERVAL)
        continue
    if not job:
        print(".", end="", flush=True)
        time.sleep(POLL_INTERVAL)
        continue
    print(f"\n🎬 Got job {job['id'][:8]} ({job['type']}, {job['style']}) — generating…")
    try:
        start_img = None
        if job["type"] == "image-to-video" and job.get("start_image_url"):
            import io as _io
            from PIL import Image as _Image
            start_img = _io.BytesIO(requests.get(job["start_image_url"], timeout=60).content)
        path = render(job["prompt"], job.get("style", "cinematic"), job["type"],
                      start_image=start_img, out_path="/tmp/output.mp4")
        report_complete(job["id"], path)
        print("✅ done, reported complete")
    except Exception as e:
        print("❌ generation failed:", e)
        report_fail(job["id"], e)
    time.sleep(5)

## ⚠️ Honest limitations (free tier)

- **Speed:** on a free T4, one ~5-second 512×768 clip takes roughly **10–25 minutes**.
- **Standalone mode:** you paste the prompt, download the MP4 yourself, and
  import it into the studio's **Library** page (or attach it to the job on the
  Dashboard). There is no automatic queue until the Vercel backend exists.
- **Sessions disconnect:** keep the tab open; if it dies mid-render, just run again.
- **Style is prompt-driven:** "anime" vs "cinematic" is a prompt suffix on the
  general LTX-Video 2B model, not a dedicated anime checkpoint.